# 01 — Mars data preparation

**Purpose:** turn a curated set of official NASA PDS Mars products into clean, aligned measurement cells for the later Mars region-discovery notebook.

This notebook does four things:

1. Resolves maintained PDS4 bundles through the PDS Search API and downloads the selected files from their archive nodes.
2. Preserves the original measurements and provenance, then applies only documented masks and coordinate conversions.
3. Produces a **0.5° medium grid** for terrain, thermal behavior, albedo, mineral abundance, and spectral shape, plus a separate **10° coarse grid** for Odyssey neutron measurements.
4. Exports both visual diagnostics for people and structured summaries for code or an AI reviewer.

It does **not** create regions, cluster the planet, detect processes, or label ice and water. Low epithermal-neutron flux is kept as a hydrogen-sensitive measurement, not converted into a claim of water ice.

## What to inspect after the run

The most useful outputs are:

- The four-map physical overview: elevation, slope, thermal inertia, and albedo.
- The mineral small multiples: they show whether composition adds meaningful spatial structure.
- The coverage and missingness plots: they show where conclusions are supported by multiple instruments.
- The PCA diagnostics: they show how much of the 73-band TES emissivity cube was preserved.
- `run_summary.md`: a short report suitable for sharing with the team.
- `mars_cells_input.parquet`: the machine-readable input to notebook 03.

Expected source download is approximately 200–250 MB. Normal execution fits comfortably within Kaggle memory and storage limits; GPUs are not needed for this preparation step.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pds4_tools": "pds4-tools==1.4",
    "pyarrow": "pyarrow>=14,<20",
}

missing_packages = [requirement for module, requirement in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_packages])

In [ ]:
import hashlib
import json
import math
import re
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import quote, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Markdown, display
from pds4_tools import read as pds4_read
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


@dataclass(frozen=True)
class Config:
    medium_pixels_per_degree: int = 2
    coarse_cell_degrees: int = 10
    terrain_source_pixels_per_degree: int = 16
    tes_source_pixels_per_degree: int = 8
    mars_radius_m: float = 3_396_000.0
    pca_variance_target: float = 0.99
    pca_training_rows: int = 50_000
    minimum_spectral_valid_fraction: float = 0.80
    mineral_error_quantile: float = 0.975
    spatial_block_degrees: int = 10
    spatial_fold_count: int = 5
    held_out_fold: int = 4
    max_total_download_gb: float = 1.0
    request_connect_timeout_s: int = 20
    request_read_timeout_s: int = 240
    random_seed: int = 42


CFG = Config()
KAGGLE_WORKING = Path("/kaggle/working")
PROJECT_ROOT = KAGGLE_WORKING / "mars_data_preparation" if KAGGLE_WORKING.exists() else Path.cwd() / "outputs" / "mars_data_preparation"
RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
TRANSFORM_DIR = PROJECT_ROOT / "transforms"
for directory in (RAW_DIR, PROCESSED_DIR, REPORT_DIR, TRANSFORM_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
RNG = np.random.default_rng(CFG.random_seed)
PDS_API_ROOT = "https://pds.nasa.gov/api/search/1"

print(f"Output root: {PROJECT_ROOT}")
print(f"Medium grid: {CFG.medium_pixels_per_degree} pixels/degree (0.5° cells)")
print(f"Coarse grid: {CFG.coarse_cell_degrees}° cells")

## Curated source manifest

PDS is an archive, not one combined table. The Search API is used to resolve maintained PDS4 bundles and preserve registry metadata. The files themselves come from the archive locations referenced by those products.

TES mineral and emissivity cubes are served by the ASU TES node, an official PDS data node. That legacy endpoint only serves these files reliably over HTTP, so every downloaded file receives a locally recorded SHA-256 digest. Maintained PDS4 MOLA and Odyssey bundles are additionally checked against their archive MD5 inventories when an entry is available.

In [ ]:
MOLA_ROOT = "https://pds-geosciences.wustl.edu/mgs/urn-nasa-pds-mgs_mola_topography_derived/meg016"
TES_SPECIAL_ROOT = "https://pds-geosciences.wustl.edu/mgs/mgs-m-tes-special-v1"
TES_NODE_ROOT = "http://tes.asu.edu/mineral_map"
GRS_ROOT = "https://pds-geosciences.wustl.edu/ody/urn-nasa-pds-odyssey_grs_special/data"

MOLA_MD5 = "https://pds-geosciences.wustl.edu/mgs/urn-nasa-pds-mgs_mola_topography_derived/urn-nasa-pds-mgs_mola_topography_derived.md5"
GRS_MD5 = "https://pds-geosciences.wustl.edu/ody/urn-nasa-pds-odyssey_grs_special/urn-nasa-pds-odyssey_grs_special.md5"

PRODUCTS = [
    {
        "product_key": "mola_topography_16ppd",
        "family": "terrain",
        "mission": "Mars Global Surveyor",
        "instrument": "MOLA",
        "variable": "median topography relative to the areoid",
        "units": "m",
        "data_url": f"{MOLA_ROOT}/megt90n000eb.img",
        "label_url": f"{MOLA_ROOT}/megt90n000eb.xml",
        "md5_index_url": MOLA_MD5,
        "pds_lid": "urn:nasa:pds:mgs_mola_topography_derived",
        "native_ppd": 16.0,
        "effective_resolution_km": 3.705,
        "expected_shape": [2880, 5760],
        "reader": "pds4_array",
    },
    {
        "product_key": "mola_counts_16ppd",
        "family": "terrain_quality",
        "mission": "Mars Global Surveyor",
        "instrument": "MOLA",
        "variable": "altimetry observations per bin",
        "units": "count",
        "data_url": f"{MOLA_ROOT}/megc90n000eb.img",
        "label_url": f"{MOLA_ROOT}/megc90n000eb.xml",
        "md5_index_url": MOLA_MD5,
        "pds_lid": "urn:nasa:pds:mgs_mola_topography_derived",
        "native_ppd": 16.0,
        "effective_resolution_km": 3.705,
        "expected_shape": [2880, 5760],
        "reader": "pds4_array",
    },
    {
        "product_key": "tes_albedo_8ppd",
        "family": "surface_reflectance",
        "mission": "Mars Global Surveyor",
        "instrument": "TES",
        "variable": "bolometric Lambert albedo",
        "units": "unitless",
        "data_url": f"{TES_SPECIAL_ROOT}/global_albedo_8ppd.img",
        "label_url": f"{TES_SPECIAL_ROOT}/global_albedo_8ppd.lbl",
        "md5_index_url": None,
        "pds_lid": None,
        "pds_dataset_id": "MGS-M-TES-SPECIAL-V1",
        "native_ppd": 8.0,
        "effective_resolution_km": 7.41,
        "expected_shape": [1440, 2880],
        "reader": "pds3_pc_real",
    },
    {
        "product_key": "tes_thermal_inertia_8ppd",
        "family": "thermophysical",
        "mission": "Mars Global Surveyor",
        "instrument": "TES",
        "variable": "thermal inertia",
        "units": "J m-2 K-1 s-1/2",
        "data_url": f"{TES_SPECIAL_ROOT}/global_ti_60n_50s_8ppd.img",
        "label_url": f"{TES_SPECIAL_ROOT}/global_ti_60n_50s_8ppd.lbl",
        "md5_index_url": None,
        "pds_lid": None,
        "pds_dataset_id": "MGS-M-TES-SPECIAL-V1",
        "native_ppd": 8.0,
        "effective_resolution_km": 7.41,
        "expected_shape": [882, 2880],
        "reader": "pds3_pc_real",
    },
    {
        "product_key": "tes_minerals_2ppd",
        "family": "composition",
        "mission": "Mars Global Surveyor",
        "instrument": "TES",
        "variable": "13 mineral, dust, and fit-error bands",
        "units": "fraction or fit error",
        "data_url": f"{TES_NODE_ROOT}/TES_mineral_maps_2_ppd.envi",
        "label_url": f"{TES_NODE_ROOT}/TES_mineral_maps_2_ppd.envi.hdr",
        "md5_index_url": None,
        "pds_lid": None,
        "pds_dataset_id": "TES global mineral maps, Bandfield 2002",
        "native_ppd": 2.0,
        "effective_resolution_km": 30.0,
        "expected_shape": [13, 360, 720],
        "reader": "envi_bsq",
    },
    {
        "product_key": "tes_emissivity_2ppd",
        "family": "spectral",
        "mission": "Mars Global Surveyor",
        "instrument": "TES",
        "variable": "73-band surface emissivity",
        "units": "unitless",
        "data_url": f"{TES_NODE_ROOT}/TES_surface_emissivity_2_ppd.envi",
        "label_url": f"{TES_NODE_ROOT}/TES_surface_emissivity_2_ppd.envi.hdr",
        "md5_index_url": None,
        "pds_lid": None,
        "pds_dataset_id": "TES global surface emissivity, Bandfield 2002",
        "native_ppd": 2.0,
        "effective_resolution_km": 30.0,
        "expected_shape": [73, 360, 720],
        "reader": "envi_bsq",
    },
]

for neutron_kind in ("epithermal", "thermal", "fast"):
    PRODUCTS.append(
        {
            "product_key": f"odyssey_ns_{neutron_kind}",
            "family": "hydrogen_sensitive",
            "mission": "2001 Mars Odyssey",
            "instrument": "Neutron Spectrometer",
            "variable": f"{neutron_kind} neutron flux",
            "units": "counts/s",
            "data_url": f"{GRS_ROOT}/ns_{neutron_kind}_020917.tab",
            "label_url": f"{GRS_ROOT}/ns_{neutron_kind}_020917.xml",
            "md5_index_url": GRS_MD5,
            "pds_lid": "urn:nasa:pds:odyssey_grs_special",
            "native_ppd": 2.0,
            "effective_resolution_km": 500.0,
            "expected_shape": [259200, 3],
            "reader": "pds_delimited_table",
        }
    )

products_df = pd.DataFrame(PRODUCTS)
display(products_df[["product_key", "family", "instrument", "native_ppd", "effective_resolution_km", "units"]])

## PDS API resolution and failure-aware download

API indexing gaps are recorded but do not incorrectly invalidate an archive-node product. A missing Search API record is a registry limitation; a failed file download, checksum mismatch, unreadable label, or invalid array shape is a real preparation failure.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": "nasa-space-apps-mars-preparation/1.0 (educational research)",
        "Accept-Encoding": "gzip, deflate",
    }
)
REQUEST_TIMEOUT = (CFG.request_connect_timeout_s, CFG.request_read_timeout_s)


def resolve_pds_product(lid: str) -> dict[str, Any]:
    endpoint = f"{PDS_API_ROOT}/products/{quote(lid, safe=':')}"
    response = SESSION.get(endpoint, headers={"Accept": "application/json"}, timeout=REQUEST_TIMEOUT)
    result = {
        "lid": lid,
        "endpoint": endpoint,
        "http_status": response.status_code,
        "resolved": response.ok,
    }
    if response.ok:
        payload = response.json()
        result["response"] = payload
        result["hits"] = payload.get("summary", {}).get("hits")
    else:
        result["error"] = response.text[:500]
    return result


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def md5_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_md5_index(url: str) -> dict[str, str]:
    response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    checksums: dict[str, str] = {}
    for raw_line in response.text.splitlines():
        match = re.match(r"^\s*([0-9a-fA-F]{32})\s+\*?(.+?)\s*$", raw_line)
        if match:
            normalized_path = match.group(2).replace("\\", "/")
            checksums[normalized_path.rsplit("/", maxsplit=1)[-1].lower()] = match.group(1).lower()
    return checksums


def local_name(url: str) -> str:
    return Path(urlparse(url).path).name


def download_file(url: str, destination: Path, byte_budget_remaining: int) -> dict[str, Any]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return {
            "url": url,
            "path": str(destination),
            "bytes": destination.stat().st_size,
            "downloaded": False,
            "http_status": "cached",
            "last_modified": None,
            "etag": None,
        }

    partial = destination.with_suffix(destination.suffix + ".part")
    existing_bytes = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={existing_bytes}-"} if existing_bytes else {}
    response = SESSION.get(url, headers=headers, stream=True, timeout=REQUEST_TIMEOUT)
    if existing_bytes and response.status_code != 206:
        partial.unlink(missing_ok=True)
        existing_bytes = 0
        response.close()
        response = SESSION.get(url, stream=True, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    reported_remaining = int(response.headers.get("Content-Length", 0))
    if reported_remaining and existing_bytes + reported_remaining > byte_budget_remaining:
        response.close()
        raise RuntimeError(f"Download budget would be exceeded by {url}")

    mode = "ab" if existing_bytes else "wb"
    downloaded_now = 0
    with partial.open(mode) as stream:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if not chunk:
                continue
            downloaded_now += len(chunk)
            if existing_bytes + downloaded_now > byte_budget_remaining:
                raise RuntimeError(f"Download budget exceeded while retrieving {url}")
            stream.write(chunk)
    partial.replace(destination)
    return {
        "url": url,
        "path": str(destination),
        "bytes": destination.stat().st_size,
        "downloaded": True,
        "http_status": response.status_code,
        "last_modified": response.headers.get("Last-Modified"),
        "etag": response.headers.get("ETag"),
    }


unique_lids = sorted({product["pds_lid"] for product in PRODUCTS if product.get("pds_lid")})
pds_resolution = [resolve_pds_product(lid) for lid in unique_lids]
(REPORT_DIR / "pds_api_resolution.json").write_text(json.dumps(pds_resolution, indent=2), encoding="utf-8")
display(pd.DataFrame([{key: item.get(key) for key in ("lid", "resolved", "http_status", "hits")} for item in pds_resolution]))

In [ ]:
md5_indexes: dict[str, dict[str, str]] = {}
for index_url in sorted({product["md5_index_url"] for product in PRODUCTS if product.get("md5_index_url")}):
    md5_indexes[index_url] = fetch_md5_index(index_url)

max_download_bytes = int(CFG.max_total_download_gb * 1024**3)
download_records: list[dict[str, Any]] = []
bytes_accounted = 0

for product in PRODUCTS:
    product_directory = RAW_DIR / product["product_key"]
    for role in ("label", "data"):
        url = product[f"{role}_url"]
        destination = product_directory / local_name(url)
        record = download_file(url, destination, max_download_bytes - bytes_accounted)
        bytes_accounted += record["bytes"]
        record.update({"product_key": product["product_key"], "role": role})
        record["sha256"] = sha256_file(destination)

        expected_md5 = None
        if product.get("md5_index_url"):
            expected_md5 = md5_indexes[product["md5_index_url"]].get(destination.name.lower())
        record["expected_md5"] = expected_md5
        record["md5_verified"] = None
        if expected_md5:
            record["md5_verified"] = md5_file(destination) == expected_md5
            if not record["md5_verified"]:
                raise RuntimeError(f"MD5 mismatch for {destination}")
        download_records.append(record)

download_manifest_columns = [
    "product_key",
    "role",
    "url",
    "path",
    "bytes",
    "downloaded",
    "http_status",
    "last_modified",
    "etag",
    "sha256",
    "expected_md5",
    "md5_verified",
]
downloads_df = pd.DataFrame(download_records).reindex(columns=download_manifest_columns)
downloads_df.to_csv(REPORT_DIR / "download_manifest.csv", index=False)
display(downloads_df[["product_key", "role", "bytes", "downloaded", "md5_verified", "sha256"]])
print(f"Source files occupy {downloads_df['bytes'].sum() / 1024**2:,.1f} MiB")

## Readers and hard validation

The loaders stop on wrong dimensions, byte order, or impossible values. They do not attempt to re-prove instrument calibration. MOLA's count layer is retained because a topographic pixel with zero shots was interpolated and should not be treated like a directly observed bin.

In [ ]:
def product_paths(product_key: str) -> tuple[Path, Path]:
    product = next(item for item in PRODUCTS if item["product_key"] == product_key)
    directory = RAW_DIR / product_key
    return directory / local_name(product["data_url"]), directory / local_name(product["label_url"])


def require_shape(array: np.ndarray, expected_shape: Iterable[int], name: str) -> np.ndarray:
    expected = tuple(int(value) for value in expected_shape)
    if array.shape == expected:
        return array
    if array.ndim == 2 and array.T.shape == expected:
        warnings.warn(f"{name} arrived transposed; axes were corrected to {expected}")
        return array.T
    raise ValueError(f"{name}: expected shape {expected}, received {array.shape}")


def load_pds4_array(product_key: str) -> np.ndarray:
    data_path, label_path = product_paths(product_key)
    product = next(item for item in PRODUCTS if item["product_key"] == product_key)
    structures = pds4_read(str(label_path), quiet=True, lazy_load=True)
    candidates = [structure for structure in structures if hasattr(structure, "data")]
    if not candidates:
        raise ValueError(f"No array structure found in {label_path}")
    structure = candidates[0]
    try:
        array = np.ma.asarray(structure.as_masked().data, dtype=np.float32).filled(np.nan)
    except (AttributeError, TypeError, ValueError):
        array = np.ma.asarray(structure.data, dtype=np.float32).filled(np.nan)
    return require_shape(np.asarray(array), product["expected_shape"], product_key)


def load_pds3_pc_real(product_key: str) -> np.ndarray:
    data_path, _ = product_paths(product_key)
    product = next(item for item in PRODUCTS if item["product_key"] == product_key)
    expected_shape = tuple(product["expected_shape"])
    array = np.fromfile(data_path, dtype="<f4")
    if array.size != math.prod(expected_shape):
        raise ValueError(f"{product_key}: expected {math.prod(expected_shape)} values, received {array.size}")
    return array.reshape(expected_shape).astype(np.float32, copy=False)


ENVI_DTYPES = {
    1: "u1",
    2: "i2",
    3: "i4",
    4: "f4",
    5: "f8",
    12: "u2",
    13: "u4",
}


def envi_header_value(text: str, key: str) -> str:
    match = re.search(rf"^\s*{re.escape(key)}\s*=\s*(.+?)\s*$", text, flags=re.IGNORECASE | re.MULTILINE)
    if not match:
        raise ValueError(f"Missing ENVI header field: {key}")
    return match.group(1).strip()


def load_envi_bsq(product_key: str) -> np.ndarray:
    data_path, header_path = product_paths(product_key)
    product = next(item for item in PRODUCTS if item["product_key"] == product_key)
    text = header_path.read_text(encoding="utf-8", errors="replace")
    samples = int(envi_header_value(text, "samples"))
    lines = int(envi_header_value(text, "lines"))
    bands = int(envi_header_value(text, "bands"))
    offset = int(envi_header_value(text, "header offset"))
    data_type_code = int(envi_header_value(text, "data type"))
    byte_order = int(envi_header_value(text, "byte order"))
    interleave = envi_header_value(text, "interleave").lower()
    if interleave != "bsq":
        raise ValueError(f"{product_key}: only the documented BSQ layout is accepted, received {interleave}")
    if data_type_code not in ENVI_DTYPES:
        raise ValueError(f"{product_key}: unsupported ENVI data type {data_type_code}")
    endian = ">" if byte_order == 1 else "<"
    dtype = np.dtype(endian + ENVI_DTYPES[data_type_code])
    expected_shape = (bands, lines, samples)
    array = np.memmap(data_path, dtype=dtype, mode="r", offset=offset, shape=expected_shape, order="C")
    require_shape(array, product["expected_shape"], product_key)
    return array


def load_neutron_table(product_key: str) -> pd.DataFrame:
    data_path, _ = product_paths(product_key)
    kind = product_key.removeprefix("odyssey_ns_")
    frame = pd.read_csv(
        data_path,
        header=None,
        names=["longitude_east", "latitude", f"neutron_{kind}_counts_s"],
        skipinitialspace=True,
    )
    expected_rows = next(item for item in PRODUCTS if item["product_key"] == product_key)["expected_shape"][0]
    if len(frame) != expected_rows:
        raise ValueError(f"{product_key}: expected {expected_rows} rows, received {len(frame)}")
    frame["longitude"] = ((frame["longitude_east"] + 180.0) % 360.0) - 180.0
    value_column = f"neutron_{kind}_counts_s"
    frame.loc[~np.isfinite(frame[value_column]) | (frame[value_column] < 0), value_column] = np.nan
    return frame[["longitude", "latitude", value_column]]


def validate_numeric_range(array: np.ndarray, name: str, lower: float, upper: float) -> dict[str, Any]:
    values = np.asarray(array, dtype=np.float64)
    finite = np.isfinite(values)
    if not finite.any():
        raise ValueError(f"{name}: no finite measurements")
    observed_min = float(np.nanmin(values))
    observed_max = float(np.nanmax(values))
    invalid = finite & ((values < lower) | (values > upper))
    invalid_fraction = float(invalid.sum() / finite.sum())
    if invalid_fraction > 0.01:
        raise ValueError(
            f"{name}: {invalid_fraction:.2%} of finite values are outside the documented range [{lower}, {upper}]"
        )
    return {
        "measurement": name,
        "finite_fraction": float(finite.mean()),
        "minimum": observed_min,
        "maximum": observed_max,
        "outside_expected_fraction": invalid_fraction,
    }

In [ ]:
mola_topography_0360 = load_pds4_array("mola_topography_16ppd").astype(np.float32)
mola_counts_0360 = load_pds4_array("mola_counts_16ppd").astype(np.float32)

# MOLA is 0–360° east; roll once to the common -180–180° domain.
mola_topography = np.roll(mola_topography_0360, mola_topography_0360.shape[1] // 2, axis=1)
mola_counts = np.roll(mola_counts_0360, mola_counts_0360.shape[1] // 2, axis=1)
del mola_topography_0360, mola_counts_0360

tes_albedo = load_pds3_pc_real("tes_albedo_8ppd")
tes_albedo[(tes_albedo < 0.0) | (tes_albedo > 1.0)] = np.nan

tes_thermal_partial = load_pds3_pc_real("tes_thermal_inertia_8ppd")
tes_thermal_partial[(tes_thermal_partial < 5.0) | (tes_thermal_partial > 5000.0)] = np.nan
tes_thermal = np.full((180 * CFG.tes_source_pixels_per_degree, 360 * CFG.tes_source_pixels_per_degree), np.nan, dtype=np.float32)
thermal_start_row = int(round((90.0 - 60.125) * CFG.tes_source_pixels_per_degree))
tes_thermal[thermal_start_row : thermal_start_row + tes_thermal_partial.shape[0], :] = tes_thermal_partial
del tes_thermal_partial

tes_minerals = np.asarray(load_envi_bsq("tes_minerals_2ppd"), dtype=np.float32)
tes_minerals[(tes_minerals < 0.0) | (tes_minerals > 1.0)] = np.nan

tes_emissivity = np.asarray(load_envi_bsq("tes_emissivity_2ppd"), dtype=np.float32)
tes_emissivity[(tes_emissivity < 0.0) | (tes_emissivity > 1.2)] = np.nan
empty_spectra = np.nansum(np.abs(tes_emissivity), axis=0) == 0
tes_emissivity[:, empty_spectra] = np.nan

neutron_tables = {
    kind: load_neutron_table(f"odyssey_ns_{kind}") for kind in ("epithermal", "thermal", "fast")
}

quality_rows = [
    validate_numeric_range(mola_topography, "MOLA elevation", -8177, 21171),
    validate_numeric_range(mola_counts, "MOLA shot counts", 0, 32767),
    validate_numeric_range(tes_albedo, "TES albedo", 0, 1),
    validate_numeric_range(tes_thermal, "TES thermal inertia", 5, 5000),
    validate_numeric_range(tes_minerals, "TES mineral cube", 0, 1),
    validate_numeric_range(tes_emissivity, "TES emissivity cube", 0, 1.2),
]
for kind, frame in neutron_tables.items():
    quality_rows.append(validate_numeric_range(frame.iloc[:, -1].to_numpy(), f"Odyssey {kind} neutron flux", 0, 1e6))

source_quality_df = pd.DataFrame(quality_rows)
source_quality_df.to_csv(REPORT_DIR / "source_quality.csv", index=False)
display(source_quality_df)

## Build neutral terrain and surface features

These features describe measurements without assigning a process:

- Elevation, relief, slope, roughness, and absolute curvature from MOLA.
- Observed-bin fraction and shot counts, so interpolation is visible to later models.
- TES albedo level, variation, and edge strength.
- TES thermal-inertia level, variation, and coverage.

Every fine source is **aggregated down** to 0.5° cells. Nothing is sharpened beyond its source resolution.
Slope and curvature are masked poleward of 88° because metric derivatives on a global cylindrical grid become unstable there. Elevation remains available, `polar_metric_limit` identifies those cells, and any polar candidate must later be refined on a polar projection.

This broad global stage intentionally uses interpretable albedo texture rather than a CNN image embedding. The available global inputs here are scalar scientific maps, not natural RGB scenes; an ImageNet embedding would introduce an opaque domain bias. Image embeddings can be benchmarked later on genuine CTX or THEMIS image patches during candidate-region refinement.

In [ ]:
def block_view(array: np.ndarray, factor: int) -> np.ndarray:
    rows, columns = array.shape
    if rows % factor or columns % factor:
        raise ValueError(f"Array shape {array.shape} is not divisible by block factor {factor}")
    return array.reshape(rows // factor, factor, columns // factor, factor)


def block_stat(array: np.ndarray, factor: int, statistic: str) -> np.ndarray:
    blocks = block_view(np.asarray(array, dtype=np.float32), factor)
    axes = (1, 3)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        if statistic == "mean":
            result = np.nanmean(blocks, axis=axes)
        elif statistic == "std":
            result = np.nanstd(blocks, axis=axes)
        elif statistic == "min":
            result = np.nanmin(blocks, axis=axes)
        elif statistic == "max":
            result = np.nanmax(blocks, axis=axes)
        elif statistic == "sum":
            result = np.nansum(blocks, axis=axes)
        elif statistic == "finite_fraction":
            result = np.isfinite(blocks).mean(axis=axes)
        elif statistic == "p90":
            result = np.nanpercentile(blocks, 90, axis=axes)
        else:
            raise ValueError(f"Unsupported statistic: {statistic}")
    return np.asarray(result, dtype=np.float32)


def spherical_gradient(array: np.ndarray, pixels_per_degree: int, radius_m: float) -> tuple[np.ndarray, np.ndarray]:
    array = np.asarray(array, dtype=np.float32)
    degree_step = 1.0 / pixels_per_degree
    latitudes = 90.0 - (np.arange(array.shape[0]) + 0.5) * degree_step
    dy = radius_m * np.deg2rad(degree_step)
    dx = radius_m * np.cos(np.deg2rad(latitudes)) * np.deg2rad(degree_step)
    dx = np.maximum(dx, dy * 0.05)
    dz_dx = (np.roll(array, -1, axis=1) - np.roll(array, 1, axis=1)) / (2.0 * dx[:, None])
    dz_dy = -np.gradient(array, dy, axis=0)
    polar_metric_limit = np.abs(latitudes) >= 88.0
    dz_dx[polar_metric_limit, :] = np.nan
    dz_dy[polar_metric_limit, :] = np.nan
    return dz_dx.astype(np.float32), dz_dy.astype(np.float32)


def terrain_features(elevation: np.ndarray, counts: np.ndarray) -> dict[str, np.ndarray]:
    factor = CFG.terrain_source_pixels_per_degree // CFG.medium_pixels_per_degree
    dz_dx, dz_dy = spherical_gradient(elevation, CFG.terrain_source_pixels_per_degree, CFG.mars_radius_m)
    slope = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype(np.float32)

    local_mean = ndimage.uniform_filter(elevation, size=3, mode=("nearest", "wrap"))
    local_square_mean = ndimage.uniform_filter(elevation**2, size=3, mode=("nearest", "wrap"))
    local_roughness = np.sqrt(np.maximum(local_square_mean - local_mean**2, 0.0)).astype(np.float32)

    d2z_dx2, _ = spherical_gradient(dz_dx, CFG.terrain_source_pixels_per_degree, CFG.mars_radius_m)
    _, d2z_dy2 = spherical_gradient(dz_dy, CFG.terrain_source_pixels_per_degree, CFG.mars_radius_m)
    curvature_abs_per_km = np.abs(d2z_dx2 + d2z_dy2) * 1000.0

    features = {
        "elevation_mean_m": block_stat(elevation, factor, "mean"),
        "elevation_std_m": block_stat(elevation, factor, "std"),
        "relief_m": block_stat(elevation, factor, "max") - block_stat(elevation, factor, "min"),
        "slope_mean_deg": block_stat(slope, factor, "mean"),
        "slope_p90_deg": block_stat(slope, factor, "p90"),
        "roughness_mean_m": block_stat(local_roughness, factor, "mean"),
        "curvature_abs_mean_per_km": block_stat(curvature_abs_per_km, factor, "mean"),
        "mola_observed_fraction": block_stat((counts > 0).astype(np.float32), factor, "mean"),
        "mola_shot_count_sum": block_stat(counts, factor, "sum"),
    }
    return features


def scalar_surface_features(array: np.ndarray, prefix: str, source_ppd: int) -> dict[str, np.ndarray]:
    factor = source_ppd // CFG.medium_pixels_per_degree
    gradient_x = (np.roll(array, -1, axis=1) - np.roll(array, 1, axis=1)) / 2.0
    gradient_y = np.gradient(array, axis=0)
    edge_strength = np.hypot(gradient_x, gradient_y).astype(np.float32)
    return {
        f"{prefix}_mean": block_stat(array, factor, "mean"),
        f"{prefix}_std": block_stat(array, factor, "std"),
        f"{prefix}_edge_mean": block_stat(edge_strength, factor, "mean"),
        f"{prefix}_coverage": block_stat(array, factor, "finite_fraction"),
    }


medium_arrays: dict[str, np.ndarray] = {}
medium_arrays.update(terrain_features(mola_topography, mola_counts))
medium_arrays.update(scalar_surface_features(tes_albedo, "albedo", CFG.tes_source_pixels_per_degree))
medium_arrays.update(scalar_surface_features(tes_thermal, "thermal_inertia", CFG.tes_source_pixels_per_degree))

expected_medium_shape = (180 * CFG.medium_pixels_per_degree, 360 * CFG.medium_pixels_per_degree)
if any(array.shape != expected_medium_shape for array in medium_arrays.values()):
    mismatches = {name: array.shape for name, array in medium_arrays.items() if array.shape != expected_medium_shape}
    raise ValueError(f"Medium-grid shape mismatch: {mismatches}")

## Add quality-aware composition and spectral-shape PCA

The original 12 mineral-abundance bands and RMS fit error remain in the output. A separate clustering copy of each mineral band is masked where the fit error is above the training geography's 97.5th percentile. This rejects the least reliable fits without turning a poor fit into a false zero abundance. The threshold is fitted without the held-out spatial fold and is exported with the transforms.

Raw emissivity PCA was dominated by overall spectral level and closely duplicated albedo. We now separate each spectrum into:

- mean emissivity level;
- spectral contrast;
- mean-centred spectral shape.

PCA is fitted only to the mean-centred shape on spatial training blocks. Missing bands are imputed using training medians solely for the PCA transform. Original emissivity values remain untouched in the downloaded ENVI cube.

In [ ]:
MINERAL_BANDS = [
    "quartz",
    "k_feldspar",
    "plagioclase",
    "amphibole",
    "low_ca_pyroxene",
    "high_ca_pyroxene",
    "olivine",
    "hematite",
    "sulfates",
    "carbonates",
    "sheet_silicates_high_si_glass",
    "surface_dust",
    "fit_rms_error",
]
MINERAL_SCIENCE_BANDS = MINERAL_BANDS[:-1]
if tes_minerals.shape[0] != len(MINERAL_BANDS):
    raise ValueError(f"Expected {len(MINERAL_BANDS)} TES mineral bands, received {tes_minerals.shape[0]}")
for band_index, band_name in enumerate(MINERAL_BANDS):
    medium_arrays[f"mineral_{band_name}"] = tes_minerals[band_index]

rows, columns = expected_medium_shape
latitude_centers = 90.0 - (np.arange(rows) + 0.5) / CFG.medium_pixels_per_degree
longitude_centers = -180.0 + (np.arange(columns) + 0.5) / CFG.medium_pixels_per_degree
longitude_grid, latitude_grid = np.meshgrid(longitude_centers, latitude_centers)

lat_block = np.floor((latitude_grid + 90.0) / CFG.spatial_block_degrees).astype(np.int16)
lon_block = np.floor((longitude_grid + 180.0) / CFG.spatial_block_degrees).astype(np.int16)
spatial_fold_grid = ((lat_block * 37 + lon_block * 17) % CFG.spatial_fold_count).astype(np.int8)

mineral_error = medium_arrays["mineral_fit_rms_error"].ravel()
mineral_training_mask = np.isfinite(mineral_error) & (spatial_fold_grid.ravel() != CFG.held_out_fold)
if mineral_training_mask.sum() < 1000:
    raise ValueError("Too few valid mineral-error measurements to fit a quality threshold")
mineral_error_threshold = float(np.nanquantile(mineral_error[mineral_training_mask], CFG.mineral_error_quantile))
mineral_science_stack = np.stack([medium_arrays[f"mineral_{name}"] for name in MINERAL_SCIENCE_BANDS])
mineral_band_coverage = np.isfinite(mineral_science_stack).mean(axis=0)
mineral_quality_valid = (
    np.isfinite(medium_arrays["mineral_fit_rms_error"])
    & (medium_arrays["mineral_fit_rms_error"] <= mineral_error_threshold)
    & (mineral_band_coverage >= 0.80)
)
medium_arrays["mineral_valid_band_fraction"] = mineral_band_coverage.astype(np.float32)
medium_arrays["mineral_quality_valid"] = mineral_quality_valid.astype(np.float32)
for band_name in MINERAL_SCIENCE_BANDS:
    medium_arrays[f"quality_masked__mineral_{band_name}"] = np.where(
        mineral_quality_valid,
        medium_arrays[f"mineral_{band_name}"],
        np.nan,
    ).astype(np.float32)

spectra = np.moveaxis(tes_emissivity, 0, -1).reshape(-1, tes_emissivity.shape[0])
spectral_valid_fraction = np.isfinite(spectra).mean(axis=1)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    emissivity_level = np.nanmean(spectra, axis=1)
    emissivity_contrast = np.nanstd(spectra, axis=1)
shape_spectra = spectra - emissivity_level[:, None]
spectral_usable = spectral_valid_fraction >= CFG.minimum_spectral_valid_fraction
train_mask = spectral_usable & (spatial_fold_grid.ravel() != CFG.held_out_fold)
train_indices = np.flatnonzero(train_mask)
if train_indices.size < 1000:
    raise ValueError(f"Only {train_indices.size} usable training spectra; PCA would be unreliable")
if train_indices.size > CFG.pca_training_rows:
    train_indices = RNG.choice(train_indices, size=CFG.pca_training_rows, replace=False)

spectral_imputer = SimpleImputer(strategy="median")
spectral_train = spectral_imputer.fit_transform(shape_spectra[train_indices])
pca = PCA(n_components=CFG.pca_variance_target, svd_solver="full")
pca.fit(spectral_train)

holdout_indices = np.flatnonzero(spectral_usable & (spatial_fold_grid.ravel() == CFG.held_out_fold))
if holdout_indices.size < 1000:
    raise ValueError(f"Only {holdout_indices.size} usable held-out spectra; PCA validation would be unreliable")
if holdout_indices.size > CFG.pca_training_rows:
    holdout_indices = RNG.choice(holdout_indices, size=CFG.pca_training_rows, replace=False)
spectral_holdout = spectral_imputer.transform(shape_spectra[holdout_indices])
spectral_holdout_reconstructed = pca.inverse_transform(pca.transform(spectral_holdout))
holdout_residual_sum_squares = float(np.square(spectral_holdout - spectral_holdout_reconstructed).sum())
holdout_total_sum_squares = float(np.square(spectral_holdout - pca.mean_).sum())
if holdout_total_sum_squares <= 0:
    raise ValueError("Held-out emissivity spectra have no measurable variation")
heldout_reconstruction_variance = 1.0 - holdout_residual_sum_squares / holdout_total_sum_squares
heldout_reconstruction_rmse = float(np.sqrt(np.mean(np.square(spectral_holdout - spectral_holdout_reconstructed))))
del spectral_holdout, spectral_holdout_reconstructed

spectral_scores = np.full((spectra.shape[0], pca.n_components_), np.nan, dtype=np.float32)
transform_indices = np.flatnonzero(spectral_usable)
transform_batch_size = 50_000
for start in range(0, transform_indices.size, transform_batch_size):
    batch_indices = transform_indices[start : start + transform_batch_size]
    batch = spectral_imputer.transform(shape_spectra[batch_indices])
    spectral_scores[batch_indices] = pca.transform(batch).astype(np.float32)

for component_index in range(pca.n_components_):
    medium_arrays[f"emissivity_shape_pc_{component_index + 1:02d}"] = spectral_scores[:, component_index].reshape(rows, columns)
medium_arrays["emissivity_level_mean"] = emissivity_level.reshape(rows, columns).astype(np.float32)
medium_arrays["emissivity_spectral_contrast"] = emissivity_contrast.reshape(rows, columns).astype(np.float32)
medium_arrays["emissivity_valid_band_fraction"] = spectral_valid_fraction.reshape(rows, columns).astype(np.float32)

pca_variance_df = pd.DataFrame(
    {
        "component": np.arange(1, pca.n_components_ + 1),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    }
)
pca_loadings_df = pd.DataFrame(
    pca.components_,
    index=[f"emissivity_shape_pc_{index + 1:02d}" for index in range(pca.n_components_)],
    columns=[f"emissivity_band_{index + 1:02d}" for index in range(tes_emissivity.shape[0])],
)
pca_variance_df.to_csv(TRANSFORM_DIR / "emissivity_shape_pca_variance.csv", index=False)
pca_loadings_df.to_csv(TRANSFORM_DIR / "emissivity_shape_pca_loadings.csv")
np.savez_compressed(
    TRANSFORM_DIR / "emissivity_shape_pca_transform.npz",
    imputer_medians=spectral_imputer.statistics_,
    pca_components=pca.components_,
    pca_mean=pca.mean_,
    explained_variance=pca.explained_variance_,
    explained_variance_ratio=pca.explained_variance_ratio_,
    singular_values=pca.singular_values_,
)
(TRANSFORM_DIR / "mineral_quality_threshold.json").write_text(
    json.dumps(
        {
            "metric": "mineral_fit_rms_error",
            "training_quantile": CFG.mineral_error_quantile,
            "threshold": mineral_error_threshold,
            "minimum_valid_band_fraction": 0.80,
            "held_out_fold": CFG.held_out_fold,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Mineral quality threshold (training q={CFG.mineral_error_quantile:.3f}): {mineral_error_threshold:.6f}")
print(f"Spectral-shape PCA retained {pca.n_components_} components for {pca.explained_variance_ratio_.sum():.2%} explained variance")
print(
    f"Held-out spectral-shape reconstruction retained {heldout_reconstruction_variance:.2%} of variation "
    f"(RMSE={heldout_reconstruction_rmse:.6f})"
)

## Assemble medium cells

Boundaries are stored numerically instead of repeated polygon text. The later region notebook can reconstruct each rectangle exactly. `parent_coarse_id` links medium cells to the legitimate 10° support used for neutron data without copying a coarse neutron value into thousands of medium rows.

In [ ]:
row_grid, column_grid = np.indices(expected_medium_shape)
medium_index = np.arange(rows * columns, dtype=np.int32)
coarse_lon_index = np.floor((longitude_grid + 180.0) / CFG.coarse_cell_degrees).astype(np.int16)
coarse_lat_index = np.floor((latitude_grid + 90.0) / CFG.coarse_cell_degrees).astype(np.int16)
parent_coarse_numeric = coarse_lat_index.astype(np.int32) * int(360 / CFG.coarse_cell_degrees) + coarse_lon_index

medium_df = pd.DataFrame(
    {
        "cell_id": [f"mars_medium_{value:06d}" for value in medium_index],
        "planet": "Mars",
        "scale": "medium_0.5deg",
        "row": row_grid.ravel().astype(np.int16),
        "column": column_grid.ravel().astype(np.int16),
        "latitude": latitude_grid.ravel().astype(np.float32),
        "longitude": longitude_grid.ravel().astype(np.float32),
        "latitude_min": (latitude_grid.ravel() - 0.25).astype(np.float32),
        "latitude_max": (latitude_grid.ravel() + 0.25).astype(np.float32),
        "longitude_min": (longitude_grid.ravel() - 0.25).astype(np.float32),
        "longitude_max": (longitude_grid.ravel() + 0.25).astype(np.float32),
        "spatial_fold": spatial_fold_grid.ravel(),
        "parent_coarse_id": [f"mars_coarse_{value:03d}" for value in parent_coarse_numeric.ravel()],
        "nominal_cell_degrees": np.float32(0.5),
        "terrain_native_resolution_km": np.float32(3.705),
        "tes_map_native_resolution_km": np.float32(7.41),
        "composition_effective_resolution_km": np.float32(30.0),
        "polar_metric_limit": (np.abs(latitude_grid.ravel()) >= 88.0),
    }
)

for feature_name, array in medium_arrays.items():
    medium_df[feature_name] = np.asarray(array, dtype=np.float32).ravel()

medium_df["missing_terrain"] = medium_df["elevation_mean_m"].isna()
medium_df["missing_thermal"] = medium_df["thermal_inertia_mean"].isna()
medium_df["missing_albedo"] = medium_df["albedo_mean"].isna()
medium_df["missing_composition"] = medium_df[[f"mineral_{name}" for name in MINERAL_BANDS[:-1]]].notna().sum(axis=1) == 0
medium_df["low_quality_composition"] = medium_df["mineral_quality_valid"].fillna(0) < 0.5
medium_df["missing_emissivity"] = medium_df["emissivity_valid_band_fraction"].fillna(0) < 0.80

medium_df["available_measurement_groups"] = (
    (~medium_df["missing_terrain"]).astype(np.int8)
    + (~medium_df["missing_thermal"]).astype(np.int8)
    + (~medium_df["missing_albedo"]).astype(np.int8)
    + (~(medium_df["missing_composition"] | medium_df["low_quality_composition"])).astype(np.int8)
    + (~medium_df["missing_emissivity"]).astype(np.int8)
)

## Assemble legitimate coarse cells and neutron measurements

Odyssey's table is sampled every 0.5°, but the instrument's physical footprint is hundreds of kilometres. The three neutron maps are therefore summarized into 10° cells. Their raw values remain neutron counts per second; no direct water-equivalent-hydrogen conversion is attempted here.

In [ ]:
def assign_coarse_id(frame: pd.DataFrame) -> pd.Series:
    lon_index = np.floor((frame["longitude"] + 180.0) / CFG.coarse_cell_degrees).astype(int).clip(0, 35)
    lat_index = np.floor((frame["latitude"] + 90.0) / CFG.coarse_cell_degrees).astype(int).clip(0, 17)
    numeric = lat_index * int(360 / CFG.coarse_cell_degrees) + lon_index
    return pd.Series([f"mars_coarse_{value:03d}" for value in numeric], index=frame.index)


coarse_feature_columns = [
    "elevation_mean_m",
    "elevation_std_m",
    "relief_m",
    "slope_mean_deg",
    "slope_p90_deg",
    "roughness_mean_m",
    "curvature_abs_mean_per_km",
    "mola_observed_fraction",
    "mola_shot_count_sum",
    "albedo_mean",
    "albedo_std",
    "albedo_edge_mean",
    "albedo_coverage",
    "thermal_inertia_mean",
    "thermal_inertia_std",
    "thermal_inertia_edge_mean",
    "thermal_inertia_coverage",
    *[f"mineral_{name}" for name in MINERAL_BANDS],
    *[f"quality_masked__mineral_{name}" for name in MINERAL_SCIENCE_BANDS],
    "mineral_valid_band_fraction",
    "mineral_quality_valid",
    *[column for column in medium_df.columns if column.startswith("emissivity_shape_pc_")],
    "emissivity_level_mean",
    "emissivity_spectral_contrast",
    "emissivity_valid_band_fraction",
]

coarse_base = medium_df.groupby("parent_coarse_id", observed=True)[coarse_feature_columns].mean(numeric_only=True)
coarse_base["mola_shot_count_sum"] = medium_df.groupby("parent_coarse_id", observed=True)["mola_shot_count_sum"].sum()
coarse_elevation_group = medium_df.groupby("parent_coarse_id", observed=True)["elevation_mean_m"]
coarse_base["relief_m"] = coarse_elevation_group.max() - coarse_elevation_group.min()
coarse_spread_columns = [
    "elevation_mean_m",
    "slope_mean_deg",
    "albedo_mean",
    "thermal_inertia_mean",
    "mineral_olivine",
    "mineral_hematite",
    "mineral_surface_dust",
]
coarse_spread = medium_df.groupby("parent_coarse_id", observed=True)[coarse_spread_columns].std(numeric_only=True)
coarse_spread = coarse_spread.add_suffix("__between_medium_std")
coarse_df = coarse_base.join(coarse_spread).reset_index().rename(columns={"parent_coarse_id": "cell_id"})

for kind, frame in neutron_tables.items():
    frame = frame.copy()
    frame["cell_id"] = assign_coarse_id(frame)
    value_column = f"neutron_{kind}_counts_s"
    grouped = frame.groupby("cell_id", observed=True)[value_column].agg(["mean", "std", "count"])
    grouped.columns = [value_column, f"{value_column}__within_cell_std", f"{value_column}__sample_count"]
    coarse_df = coarse_df.merge(grouped.reset_index(), on="cell_id", how="left", validate="one_to_one")

coarse_numeric_id = coarse_df["cell_id"].str.extract(r"(\d+)$", expand=False).astype(int)
coarse_columns_per_row = int(360 / CFG.coarse_cell_degrees)
coarse_lat_index_values = coarse_numeric_id // coarse_columns_per_row
coarse_lon_index_values = coarse_numeric_id % coarse_columns_per_row
coarse_df["planet"] = "Mars"
coarse_df["scale"] = "coarse_10deg"
coarse_df["row"] = coarse_lat_index_values.astype(np.int16)
coarse_df["column"] = coarse_lon_index_values.astype(np.int16)
coarse_df["latitude_min"] = -90.0 + coarse_lat_index_values * CFG.coarse_cell_degrees
coarse_df["latitude_max"] = coarse_df["latitude_min"] + CFG.coarse_cell_degrees
coarse_df["longitude_min"] = -180.0 + coarse_lon_index_values * CFG.coarse_cell_degrees
coarse_df["longitude_max"] = coarse_df["longitude_min"] + CFG.coarse_cell_degrees
coarse_df["latitude"] = (coarse_df["latitude_min"] + coarse_df["latitude_max"]) / 2.0
coarse_df["longitude"] = (coarse_df["longitude_min"] + coarse_df["longitude_max"]) / 2.0
coarse_df["spatial_fold"] = (
    (coarse_lat_index_values * 37 + coarse_lon_index_values * 17) % CFG.spatial_fold_count
).astype(np.int8)
coarse_df["parent_coarse_id"] = pd.NA
coarse_df["nominal_cell_degrees"] = np.float32(CFG.coarse_cell_degrees)
coarse_df["terrain_native_resolution_km"] = np.float32(3.705)
coarse_df["tes_map_native_resolution_km"] = np.float32(7.41)
coarse_df["composition_effective_resolution_km"] = np.float32(30.0)
coarse_df["neutron_effective_resolution_km"] = np.float32(500.0)
coarse_df["polar_metric_limit"] = (coarse_df["latitude_max"] > 88.0) | (coarse_df["latitude_min"] < -88.0)

coarse_df["missing_terrain"] = coarse_df["elevation_mean_m"].isna()
coarse_df["missing_thermal"] = coarse_df["thermal_inertia_mean"].isna()
coarse_df["missing_albedo"] = coarse_df["albedo_mean"].isna()
coarse_df["missing_composition"] = coarse_df[[f"mineral_{name}" for name in MINERAL_BANDS[:-1]]].isna().all(axis=1)
coarse_df["low_quality_composition"] = coarse_df[
    [f"quality_masked__mineral_{name}" for name in MINERAL_SCIENCE_BANDS]
].isna().all(axis=1)
coarse_df["missing_emissivity"] = coarse_df["emissivity_valid_band_fraction"].fillna(0) < 0.80
coarse_df["missing_neutron"] = coarse_df[
    ["neutron_epithermal_counts_s", "neutron_thermal_counts_s", "neutron_fast_counts_s"]
].isna().all(axis=1)
coarse_df["available_measurement_groups"] = (
    (~coarse_df["missing_terrain"]).astype(np.int8)
    + (~coarse_df["missing_thermal"]).astype(np.int8)
    + (~coarse_df["missing_albedo"]).astype(np.int8)
    + (~(coarse_df["missing_composition"] | coarse_df["low_quality_composition"])).astype(np.int8)
    + (~coarse_df["missing_emissivity"]).astype(np.int8)
    + (~coarse_df["missing_neutron"]).astype(np.int8)
).astype(np.int8)
medium_df["missing_neutron"] = True

print(f"Medium cells: {len(medium_df):,}")
print(f"Coarse cells: {len(coarse_df):,}")

## Explicit feature roles, robust scaling, and group balance

Direct measurements stay in their physical units. The schema below prevents acquisition quality from silently entering clustering:

- `science_default`: used by the recommended clustering representation.
- `science_optional`: scientifically meaningful but excluded by default because it is redundant or answers a narrower question.
- `quality_context`: retained for artifact tests and confidence, never used to define the default clusters.
- `geographic_context`: retained for spatial validation, never used to define the default clusters.

Default science features are scaled separately for each scale using statistics fitted only on non-held-out spatial blocks. The normal denominator is the interquartile range. Sparse, zero-heavy variables use a wider central percentile span when their IQR is zero; a truly constant or unsupported column is recorded and excluded. Active features are then multiplied by `1 / sqrt(number of active features in their group)`. This stops a 12-band composition group from dominating merely because it has more columns than terrain.

In [ ]:
feature_schema_rows: list[dict[str, Any]] = []


def register_feature(scale: str, feature: str, group: str, role: str, reason: str) -> None:
    feature_schema_rows.append(
        {
            "scale": scale,
            "feature": feature,
            "group": group,
            "role": role,
            "default_include": role == "science_default",
            "reason": reason,
        }
    )


for scale_name in ("medium_0.5deg", "coarse_10deg"):
    for feature in ("elevation_mean_m", "relief_m", "slope_mean_deg"):
        register_feature(scale_name, feature, "terrain", "science_default", "Complementary terrain level, range, or gradient")
    for feature in ("elevation_std_m", "slope_p90_deg", "roughness_mean_m", "curvature_abs_mean_per_km"):
        register_feature(scale_name, feature, "terrain", "science_optional", "Useful detail but redundant with default terrain variables")
    for feature in ("thermal_inertia_mean", "thermal_inertia_std"):
        register_feature(scale_name, feature, "thermophysical", "science_default", "Surface thermal level and local variability")
    register_feature(scale_name, "thermal_inertia_edge_mean", "thermophysical", "science_optional", "Boundary-sensitive thermal detail")
    for feature in ("albedo_mean", "albedo_std"):
        register_feature(scale_name, feature, "reflectance", "science_default", "Surface reflectance level and local variability")
    register_feature(scale_name, "albedo_edge_mean", "reflectance", "science_optional", "Boundary-sensitive reflectance detail")
    for band_name in MINERAL_SCIENCE_BANDS:
        register_feature(
            scale_name,
            f"quality_masked__mineral_{band_name}",
            "composition",
            "science_default",
            "Mineral abundance retained only where the spectral fit passes the training-derived quality mask",
        )
    register_feature(scale_name, "emissivity_spectral_contrast", "spectral_shape", "science_default", "Strength of spectral variation independent of mean level")
    for component_index in range(pca.n_components_):
        register_feature(
            scale_name,
            f"emissivity_shape_pc_{component_index + 1:02d}",
            "spectral_shape",
            "science_default",
            "Mean-centred TES emissivity shape component",
        )
    register_feature(scale_name, "emissivity_level_mean", "spectral_level", "science_optional", "Excluded by default because it strongly overlaps albedo")

for feature in ("neutron_epithermal_counts_s", "neutron_thermal_counts_s", "neutron_fast_counts_s"):
    register_feature("coarse_10deg", feature, "hydrogen_sensitive", "science_default", "Coarse neutron-flux measurement at a defensible physical scale")

quality_context_features = [
    "mola_observed_fraction",
    "mola_shot_count_sum",
    "albedo_coverage",
    "thermal_inertia_coverage",
    "mineral_fit_rms_error",
    "mineral_valid_band_fraction",
    "mineral_quality_valid",
    "emissivity_valid_band_fraction",
    "available_measurement_groups",
    "missing_terrain",
    "missing_thermal",
    "missing_albedo",
    "missing_composition",
    "low_quality_composition",
    "missing_emissivity",
    "missing_neutron",
    "polar_metric_limit",
]
geographic_context_features = [
    "row",
    "column",
    "latitude",
    "longitude",
    "latitude_min",
    "latitude_max",
    "longitude_min",
    "longitude_max",
    "spatial_fold",
    "nominal_cell_degrees",
    "terrain_native_resolution_km",
    "tes_map_native_resolution_km",
    "composition_effective_resolution_km",
    "neutron_effective_resolution_km",
]
for scale_name, frame in (("medium_0.5deg", medium_df), ("coarse_10deg", coarse_df)):
    for feature in quality_context_features:
        if feature in frame.columns:
            register_feature(scale_name, feature, "quality", "quality_context", "Artifact, coverage, or confidence diagnostic")
    for feature in geographic_context_features:
        if feature in frame.columns:
            register_feature(scale_name, feature, "geography", "geographic_context", "Spatial validation or resolution metadata")

feature_schema_df = pd.DataFrame(feature_schema_rows).drop_duplicates(subset=["scale", "feature"], keep="first")


def build_scaling_plan(
    frame: pd.DataFrame,
    scale_name: str,
    schema: pd.DataFrame,
) -> pd.DataFrame:
    """Choose a training-only scale that also works for sparse, zero-heavy variables."""
    transform_rows: list[dict[str, Any]] = []
    training = frame["spatial_fold"] != CFG.held_out_fold
    scale_schema = schema.loc[(schema["scale"] == scale_name) & schema["default_include"]]
    missing_features = sorted(set(scale_schema["feature"]) - set(frame.columns))
    if missing_features:
        raise ValueError(f"{scale_name}: schema features missing from the table: {missing_features}")
    for row in scale_schema.itertuples(index=False):
        column = row.feature
        train_values = pd.to_numeric(frame.loc[training, column], errors="coerce")
        finite_values = train_values[np.isfinite(train_values)]
        statistics = {
            "median": np.nan,
            "q01": np.nan,
            "q05": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "q95": np.nan,
            "q99": np.nan,
            "standard_deviation": np.nan,
            "minimum": np.nan,
            "maximum": np.nan,
        }
        active = len(finite_values) >= 50
        inactive_reason: str | None = None if active else "fewer_than_50_finite_training_values"
        scaling_method: str | None = None
        denominator = np.nan
        if active:
            quantiles = finite_values.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
            statistics = {
                "median": float(quantiles.loc[0.50]),
                "q01": float(quantiles.loc[0.01]),
                "q05": float(quantiles.loc[0.05]),
                "q25": float(quantiles.loc[0.25]),
                "q75": float(quantiles.loc[0.75]),
                "q95": float(quantiles.loc[0.95]),
                "q99": float(quantiles.loc[0.99]),
                "standard_deviation": float(finite_values.std(ddof=0)),
                "minimum": float(finite_values.min()),
                "maximum": float(finite_values.max()),
            }
            scale_candidates = [
                ("interquartile_range", statistics["q75"] - statistics["q25"]),
                ("central_90_percent_span", statistics["q95"] - statistics["q05"]),
                ("central_98_percent_span", statistics["q99"] - statistics["q01"]),
                ("standard_deviation_sparse_fallback", statistics["standard_deviation"]),
            ]
            for candidate_method, candidate_denominator in scale_candidates:
                if np.isfinite(candidate_denominator) and candidate_denominator > 1e-12:
                    scaling_method = candidate_method
                    denominator = float(candidate_denominator)
                    break
            if scaling_method is None:
                active = False
                inactive_reason = "constant_in_training_geography"
        transform_rows.append(
            {
                "scale": scale_name,
                "feature": column,
                "group": row.group,
                "training_rows": int(len(finite_values)),
                "active_for_model": active,
                "inactive_reason": inactive_reason,
                "scaling_method": scaling_method,
                "scale_denominator": denominator,
                **statistics,
                "held_out_fold": CFG.held_out_fold,
            }
        )
    return pd.DataFrame(transform_rows)


medium_plan_df = build_scaling_plan(medium_df, "medium_0.5deg", feature_schema_df)
coarse_plan_df = build_scaling_plan(coarse_df, "coarse_10deg", feature_schema_df)
scaling_transforms_df = pd.concat([medium_plan_df, coarse_plan_df], ignore_index=True)

requested_groups = set(
    feature_schema_df.loc[feature_schema_df["default_include"], ["scale", "group"]]
    .itertuples(index=False, name=None)
)
active_groups = set(
    scaling_transforms_df.loc[scaling_transforms_df["active_for_model"], ["scale", "group"]]
    .itertuples(index=False, name=None)
)
missing_groups = sorted(requested_groups - active_groups)
if missing_groups:
    raise ValueError(f"No usable default feature remains for these scale/group pairs: {missing_groups}")

active_counts = (
    scaling_transforms_df.loc[scaling_transforms_df["active_for_model"]]
    .groupby(["scale", "group"], observed=True)["feature"]
    .transform("count")
)
scaling_transforms_df.loc[scaling_transforms_df["active_for_model"], "group_feature_count"] = active_counts.astype(int)
scaling_transforms_df["group_weight"] = np.where(
    scaling_transforms_df["active_for_model"],
    1.0 / np.sqrt(scaling_transforms_df["group_feature_count"].astype(float)),
    np.nan,
)

feature_schema_df = feature_schema_df.merge(
    scaling_transforms_df[
        [
            "scale",
            "feature",
            "active_for_model",
            "inactive_reason",
            "scaling_method",
            "scale_denominator",
            "group_feature_count",
            "group_weight",
        ]
    ],
    on=["scale", "feature"],
    how="left",
    validate="one_to_one",
)
feature_schema_df["active_for_model"] = feature_schema_df["active_for_model"].eq(True)
feature_schema_df["scaled_column"] = np.where(
    feature_schema_df["active_for_model"],
    "scaled__" + feature_schema_df["feature"],
    pd.NA,
)
feature_schema_df["model_column"] = np.where(
    feature_schema_df["active_for_model"],
    "model__" + feature_schema_df["feature"],
    pd.NA,
)


def apply_scaling_plan(frame: pd.DataFrame, scale_name: str, plan: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    active_plan = plan.loc[(plan["scale"] == scale_name) & plan["active_for_model"]]
    for row in active_plan.itertuples(index=False):
        values = pd.to_numeric(result[row.feature], errors="coerce")
        scaled_column = f"scaled__{row.feature}"
        model_column = f"model__{row.feature}"
        result[scaled_column] = ((values - float(row.median)) / float(row.scale_denominator)).astype(np.float32)
        result[model_column] = (result[scaled_column] * float(row.group_weight)).astype(np.float32)
    return result


medium_scaled_df = apply_scaling_plan(medium_df, "medium_0.5deg", scaling_transforms_df)
coarse_scaled_df = apply_scaling_plan(coarse_df, "coarse_10deg", scaling_transforms_df)
scaling_transforms_df.to_csv(TRANSFORM_DIR / "robust_scaling_parameters.csv", index=False)
feature_schema_df.to_csv(REPORT_DIR / "feature_schema.csv", index=False)
(REPORT_DIR / "feature_schema.json").write_text(
    feature_schema_df.where(pd.notna(feature_schema_df), None).to_json(orient="records", indent=2),
    encoding="utf-8",
)

all_cells_df = pd.concat([medium_scaled_df, coarse_scaled_df], ignore_index=True, sort=False)
all_cells_df["is_spatial_holdout"] = all_cells_df["spatial_fold"] == CFG.held_out_fold

display(
    feature_schema_df.groupby(["scale", "role", "group"], observed=True)
    .size()
    .rename("feature_count")
    .reset_index()
)
display(
    scaling_transforms_df.groupby(
        ["scale", "active_for_model", "scaling_method"],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("feature_count")
    .reset_index()
)

## Visual analysis dashboard

These figures answer specific questions rather than decorating the notebook:

1. Do major physical measurements show coherent and different spatial structure?
2. Which mineral variables add information, and where is the spectral fit weak?
3. Where is the dataset incomplete?
4. Did PCA preserve the emissivity cube compactly, and are major model features redundant?

In [ ]:
def feature_raster(frame: pd.DataFrame, feature: str) -> np.ndarray:
    ordered = frame.sort_values(["row", "column"])
    return ordered[feature].to_numpy().reshape(expected_medium_shape)


def robust_limits(array: np.ndarray, low: float = 2.0, high: float = 98.0) -> tuple[float, float]:
    finite = np.asarray(array)[np.isfinite(array)]
    return tuple(np.percentile(finite, [low, high]).tolist())


def draw_global_map(ax: plt.Axes, array: np.ndarray, title: str, cmap: str, colorbar_label: str) -> None:
    lower, upper = robust_limits(array)
    image = ax.imshow(
        array,
        extent=[-180, 180, -90, 90],
        origin="upper",
        aspect="auto",
        cmap=cmap,
        vmin=lower,
        vmax=upper,
        interpolation="nearest",
    )
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    colorbar = plt.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
    colorbar.set_label(colorbar_label)


physical_maps = [
    ("elevation_mean_m", "MOLA elevation", "terrain", "metres"),
    ("slope_mean_deg", "Mean terrain slope", "magma", "degrees"),
    ("thermal_inertia_mean", "TES thermal inertia", "inferno", "J m⁻² K⁻¹ s⁻¹ᐟ²"),
    ("albedo_mean", "TES bolometric albedo", "gray", "unitless"),
]
fig, axes = plt.subplots(2, 2, figsize=(18, 9), constrained_layout=True)
for axis, (feature, title, cmap, label) in zip(axes.ravel(), physical_maps):
    draw_global_map(axis, feature_raster(medium_df, feature), title, cmap, label)
fig.suptitle("Mars physical overview — aligned 0.5° cells", fontsize=18, fontweight="bold")
plt.show()

In [ ]:
mineral_maps = [
    ("quality_masked__mineral_olivine", "Olivine — quality masked", "viridis"),
    ("quality_masked__mineral_high_ca_pyroxene", "High-Ca pyroxene — quality masked", "viridis"),
    ("quality_masked__mineral_hematite", "Hematite — quality masked", "rocket"),
    ("quality_masked__mineral_sulfates", "Sulfates — quality masked", "mako"),
    ("quality_masked__mineral_surface_dust", "Surface dust — quality masked", "copper"),
    ("mineral_fit_rms_error", "Mineral-model RMS error", "flare"),
]
fig, axes = plt.subplots(3, 2, figsize=(18, 12), constrained_layout=True)
for axis, (feature, title, cmap) in zip(axes.ravel(), mineral_maps):
    draw_global_map(axis, feature_raster(medium_df, feature), title, cmap, "fraction or fit error")
fig.suptitle("TES composition — usable science values and fit error", fontsize=18, fontweight="bold")
plt.show()

In [ ]:
coverage_columns = {
    "Terrain": ~medium_df["missing_terrain"],
    "Thermal": ~medium_df["missing_thermal"],
    "Albedo": ~medium_df["missing_albedo"],
    "Minerals": ~(medium_df["missing_composition"] | medium_df["low_quality_composition"]),
    "Emissivity": ~medium_df["missing_emissivity"],
}
coverage_df = pd.DataFrame(
    {"measurement_group": list(coverage_columns), "coverage_percent": [100.0 * values.mean() for values in coverage_columns.values()]}
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
sns.barplot(data=coverage_df, x="coverage_percent", y="measurement_group", ax=axes[0], color="#3B82F6")
axes[0].set_xlim(0, 100)
axes[0].set_xlabel("Medium cells with usable data (%)")
axes[0].set_ylabel("")
axes[0].set_title("Coverage by measurement family", loc="left", fontweight="bold")

availability = feature_raster(medium_df, "available_measurement_groups")
image = axes[1].imshow(
    availability,
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap="viridis",
    vmin=0,
    vmax=5,
    interpolation="nearest",
)
axes[1].set_title("Number of available medium-scale families", loc="left", fontweight="bold")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(image, ax=axes[1], fraction=0.025, pad=0.02, label="available families")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
sns.lineplot(
    data=pca_variance_df,
    x="component",
    y="cumulative_explained_variance",
    marker="o",
    ax=axes[0],
    color="#7C3AED",
)
axes[0].axhline(CFG.pca_variance_target, color="#DC2626", linestyle="--", label=f"target {CFG.pca_variance_target:.0%}")
axes[0].set_ylim(0, 1.01)
axes[0].set_title(
    f"Spectral shape retained — held-out: {heldout_reconstruction_variance:.1%}",
    loc="left",
    fontweight="bold",
)
axes[0].set_ylabel("Cumulative explained variance")
axes[0].legend()

loading_subset = pca_loadings_df.iloc[: min(6, len(pca_loadings_df)), :]
sns.heatmap(
    loading_subset,
    cmap="vlag",
    center=0,
    xticklabels=8,
    yticklabels=True,
    ax=axes[1],
    cbar_kws={"label": "PCA loading"},
)
axes[1].set_title("First shape components across 73 TES bands", loc="left", fontweight="bold")
axes[1].set_xlabel("Emissivity band index")
axes[1].set_ylabel("")
plt.show()

In [ ]:
correlation_features = [
    "elevation_mean_m",
    "slope_mean_deg",
    "roughness_mean_m",
    "thermal_inertia_mean",
    "albedo_mean",
    "mineral_olivine",
    "mineral_hematite",
    "mineral_surface_dust",
    "mineral_fit_rms_error",
    "emissivity_level_mean",
    "emissivity_shape_pc_01",
]
correlation_features = [feature for feature in correlation_features if feature in medium_df.columns]
correlation_sample = medium_df.loc[:, correlation_features].sample(
    n=min(40_000, len(medium_df)), random_state=CFG.random_seed
)
correlation = correlation_sample.corr(method="spearman", min_periods=1000)

plt.figure(figsize=(11, 9))
sns.heatmap(correlation, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, annot=True, fmt=".2f")
plt.title("Raw-feature diagnostic — redundancy and quality coupling", loc="left", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## Export reproducible artifacts and a short team report

The Parquet table contains both physical-unit and scaled columns. Separate CSV/JSON files preserve provenance, quality checks, PCA loadings, and scaling parameters so no transform is hidden inside the notebook state.

In [ ]:
cells_path = PROCESSED_DIR / "mars_cells_input.parquet"
medium_path = PROCESSED_DIR / "mars_medium_cells.parquet"
coarse_path = PROCESSED_DIR / "mars_coarse_cells.parquet"
aligned_layers_path = PROCESSED_DIR / "mars_medium_aligned_layers.npz"
aligned_masks_path = PROCESSED_DIR / "mars_medium_masks.npz"
provenance_path = REPORT_DIR / "provenance_manifest.json"

all_cells_df.to_parquet(cells_path, index=False, compression="zstd")
medium_scaled_df.to_parquet(medium_path, index=False, compression="zstd")
coarse_scaled_df.to_parquet(coarse_path, index=False, compression="zstd")
np.savez_compressed(aligned_layers_path, **medium_arrays)
np.savez_compressed(
    aligned_masks_path,
    missing_terrain=medium_df["missing_terrain"].to_numpy().reshape(expected_medium_shape),
    missing_thermal=medium_df["missing_thermal"].to_numpy().reshape(expected_medium_shape),
    missing_albedo=medium_df["missing_albedo"].to_numpy().reshape(expected_medium_shape),
    missing_composition=medium_df["missing_composition"].to_numpy().reshape(expected_medium_shape),
    low_quality_composition=medium_df["low_quality_composition"].to_numpy().reshape(expected_medium_shape),
    missing_emissivity=medium_df["missing_emissivity"].to_numpy().reshape(expected_medium_shape),
)

provenance_records = []
for product in PRODUCTS:
    product_downloads = downloads_df.loc[downloads_df["product_key"] == product["product_key"]]
    downloaded_files = product_downloads.reindex(
        columns=[
            "role",
            "url",
            "path",
            "bytes",
            "sha256",
            "expected_md5",
            "md5_verified",
            "last_modified",
            "etag",
        ]
    )
    downloaded_files = downloaded_files.astype(object).where(downloaded_files.notna(), None)
    provenance_records.append(
        {
            **product,
            "downloaded_files": downloaded_files.to_dict(orient="records"),
            "processing": {
                "longitude_domain": "-180 to 180 degrees east",
                "latitude_type": "planetocentric",
                "medium_grid_degrees": 1.0 / CFG.medium_pixels_per_degree,
                "coarse_grid_degrees": CFG.coarse_cell_degrees,
                "aggregation": "block summaries; no sharpening",
                "composition_quality": "science bands masked using a training-fold mineral-fit RMS threshold",
                "emissivity_representation": "mean level retained separately; PCA fitted to mean-centred spectral shape",
                "clustering_representation": "robust-scaled default science features with equalized group influence",
            },
        }
    )
provenance_path.write_text(json.dumps(provenance_records, indent=2, allow_nan=False), encoding="utf-8")

coverage_summary = {
    name: round(100.0 * values.mean(), 2) for name, values in coverage_columns.items()
}
official_md5_available = int(downloads_df["expected_md5"].notna().sum())
official_md5_verified = int(downloads_df["md5_verified"].eq(True).sum())
default_feature_counts = (
    feature_schema_df.loc[feature_schema_df["active_for_model"]]
    .groupby("scale", observed=True)["feature"]
    .nunique()
    .astype(int)
    .to_dict()
)
inactive_default_features = feature_schema_df.loc[
    feature_schema_df["default_include"] & ~feature_schema_df["active_for_model"],
    ["scale", "feature", "inactive_reason"],
].to_dict(orient="records")
mineral_quality_retained_percent = round(100.0 * medium_df["mineral_quality_valid"].fillna(0).mean(), 2)
summary = {
    "status": "complete",
    "run_started_utc": RUN_STARTED_UTC,
    "run_finished_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": asdict(CFG),
    "source_download_mib": round(float(downloads_df["bytes"].sum() / 1024**2), 2),
    "source_products": len(PRODUCTS),
    "pds4_bundles_resolved": int(sum(bool(item["resolved"]) for item in pds_resolution)),
    "pds4_bundles_requested": len(pds_resolution),
    "medium_cells": int(len(medium_scaled_df)),
    "coarse_cells": int(len(coarse_scaled_df)),
    "total_rows": int(len(all_cells_df)),
    "physical_and_context_columns": int(
        sum(not column.startswith(("scaled__", "model__")) for column in all_cells_df.columns)
    ),
    "scaled_columns": int(sum(column.startswith("scaled__") for column in all_cells_df.columns)),
    "model_columns": int(sum(column.startswith("model__") for column in all_cells_df.columns)),
    "default_model_features_by_scale": default_feature_counts,
    "inactive_default_features": inactive_default_features,
    "official_md5_available": official_md5_available,
    "official_md5_verified": official_md5_verified,
    "coverage_percent": coverage_summary,
    "mineral_quality_retained_percent": mineral_quality_retained_percent,
    "emissivity_input_bands": int(tes_emissivity.shape[0]),
    "emissivity_shape_pca_components": int(pca.n_components_),
    "emissivity_shape_variance_retained": float(pca.explained_variance_ratio_.sum()),
    "emissivity_shape_heldout_reconstruction_variance": heldout_reconstruction_variance,
    "emissivity_shape_heldout_reconstruction_rmse": heldout_reconstruction_rmse,
    "held_out_spatial_fold": CFG.held_out_fold,
    "important_cautions": [
        "Neutron flux is hydrogen-sensitive evidence, not a direct water-ice label.",
        "MOLA bins with zero shots contain interpolated elevation; use mola_observed_fraction.",
        "Thermal inertia covers a restricted latitude range, so missingness is meaningful.",
        "Mineral abundances are model-derived; default clustering uses only training-threshold quality-masked copies.",
        "Quality, coverage, coordinates, and resolution fields are context for validation, not default cluster inputs.",
        "Slope and curvature are masked poleward of 88 degrees; polar refinement requires a polar projection.",
        "Medium and coarse rows are separate scales and must not be treated as independent copies of one measurement.",
    ],
    "outputs": {
        "combined_cells": str(cells_path),
        "medium_cells": str(medium_path),
        "coarse_cells": str(coarse_path),
        "aligned_medium_layers": str(aligned_layers_path),
        "aligned_medium_masks": str(aligned_masks_path),
        "provenance": str(provenance_path),
        "download_manifest": str(REPORT_DIR / "download_manifest.csv"),
        "source_quality": str(REPORT_DIR / "source_quality.csv"),
        "mineral_quality_threshold": str(TRANSFORM_DIR / "mineral_quality_threshold.json"),
        "pca_variance": str(TRANSFORM_DIR / "emissivity_shape_pca_variance.csv"),
        "pca_loadings": str(TRANSFORM_DIR / "emissivity_shape_pca_loadings.csv"),
        "pca_transform": str(TRANSFORM_DIR / "emissivity_shape_pca_transform.npz"),
        "scaling_parameters": str(TRANSFORM_DIR / "robust_scaling_parameters.csv"),
        "feature_schema_csv": str(REPORT_DIR / "feature_schema.csv"),
        "feature_schema_json": str(REPORT_DIR / "feature_schema.json"),
    },
}

(REPORT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary_markdown = f"""# Mars preparation run summary

- Status: **{summary['status']}**
- Sources: **{summary['source_products']} products**, {summary['source_download_mib']:.1f} MiB on disk
- Rows: **{summary['medium_cells']:,} medium cells** and **{summary['coarse_cells']:,} coarse cells**
- TES emissivity shape: **{summary['emissivity_input_bands']} bands → {summary['emissivity_shape_pca_components']} PCA components**, preserving {summary['emissivity_shape_variance_retained']:.2%} of training variance
- Held-out spectral reconstruction: **{summary['emissivity_shape_heldout_reconstruction_variance']:.2%} of variation retained** (RMSE {summary['emissivity_shape_heldout_reconstruction_rmse']:.6f})
- Default clustering features: **{default_feature_counts['medium_0.5deg']} medium-scale** and **{default_feature_counts['coarse_10deg']} coarse-scale**
- Official checksums: **{official_md5_verified}/{official_md5_available} verified** where the archive supplied MD5 values
- Terrain coverage: **{coverage_summary['Terrain']:.1f}%**
- Thermal coverage: **{coverage_summary['Thermal']:.1f}%**
- Quality-passing mineral coverage: **{coverage_summary['Minerals']:.1f}%** ({mineral_quality_retained_percent:.1f}% of all medium cells)
- Emissivity coverage: **{coverage_summary['Emissivity']:.1f}%**

## Interpretation limits

1. Neutron measurements are evidence sensitive to hydrogen, not proof of ice.
2. MOLA observation coverage must accompany topography because empty bins were interpolated.
3. TES mineral values are model results; default clustering receives only the quality-masked copies.
4. Slope and curvature are masked poleward of 88°; polar candidates need a polar-projected refinement.
5. The 0.5° and 10° rows represent different physical scales and should be analysed separately.
6. Quality, coverage, coordinates, and resolution are validation context, not default clustering inputs.
7. This notebook prepared measurements only. Region creation and process interpretation belong to notebook 03.
"""
(REPORT_DIR / "run_summary.md").write_text(summary_markdown, encoding="utf-8")

display(Markdown(summary_markdown))
display(pd.DataFrame([{"artifact": key, "path": value} for key, value in summary["outputs"].items()]))

## Handoff to notebook 03

`mars_cells_input.parquet` is ready for region discovery when all hard checks above pass. Notebook 03 must:

- analyse `medium_0.5deg` and `coarse_10deg` separately;
- create connected homogeneous regions before clustering;
- use only the `model__*` columns with `active_for_model = true` in `feature_schema.csv`;
- keep raw physical values available beside the model representation for interpretation;
- test whether instrument metadata, missingness, latitude, or resolution predicts clusters;
- use scientific literature only after neutral region types are stable;
- treat unusual regions as anomalies requiring review, not automatic process discoveries.